In [1]:
#Question 1
import re

ZIP_RE = re.compile(r"\b\d{5}(?:[-\s]\d{4})?\b")
NO_CAP_WORD_RE = re.compile(r"\b(?![A-Z])[A-Za-z]+(?:['-][A-Za-z]+)*\b")
NUMBER_RE = re.compile(r"[+-]?(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?")
EMAIL_RE = re.compile(r"(?i)\be(?:[-–\s])?mail\b")
GO_RE = re.compile(r"\bgo+\b[!.,?]?")
QUESTION_LINE_RE = re.compile(r"^.*\?(?:[)\"”’\]]\s*)*$")

text = """
ZIPs: 12345, 12345-6789, 12345 6789, invalid 9123456
Words: Apple apple don't state-of-the-art
Nums: -1, 1,234.56, +7.0e-3, 42E2
Email variants: email, e-mail, e mail, E–MAIL
Interjections: go goo gooo! go?
Line ends? yes?  " )
No question here.
"""

print("ZIP matches:", ZIP_RE.findall(text))
print("Words not starting with capital:", NO_CAP_WORD_RE.findall(text))
print("Numbers:", NUMBER_RE.findall(text))
print("Email matches:", EMAIL_RE.findall(text))
print("Go matches:", GO_RE.findall(text))

print("\nLines ending with ? (including closing quotes/brackets):")
for line in text.splitlines():
    if QUESTION_LINE_RE.match(line):
        print("  ", repr(line))



ZIP matches: ['12345', '12345-6789', '12345 6789']
Words not starting with capital: ['invalid', 'apple', "don't", 'state-of-the-art', 'variants', 'email', 'e-mail', 'e', 'mail', 'go', 'goo', 'gooo', 'go', 'ends', 'yes', 'question', 'here']
Numbers: ['123', '45', '123', '45', '-678', '9', '123', '45', '678', '9', '912', '345', '6', '-1', '1,234.56', '+7.0e-3', '42E2']
Email matches: ['Email', 'email', 'e-mail', 'e mail', 'E–MAIL']
Go matches: ['go', 'goo', 'gooo!', 'go?']

Lines ending with ? (including closing quotes/brackets):
   'Interjections: go goo gooo! go?'


#Question 2.1

Byte Pair Encoding Training

The training corpus contains repeated words which are initially split into individual characters with end-of-word marker _.

Initial Tokenization

low → l o w _

lowest → l o w e s t _

newer → n e w e r _

wider → w i d e r _

Pair Frequency Calculation

After counting adjacent symbol pairs, the most frequent pairs were identified as:

e r → highest frequency

er _ → next highest frequency

n e → next highest frequency

Merge Operations

Merge 1

e r → er


Merge 2

er _ → er_


Merge 3

n e → ne

Final Tokens

The final subword units created help reduce vocabulary size and allow unseen words to be represented effectively.

#Question 2.2

A Python program was developed to simulate the BPE learning process. The program identifies the most frequent pair of characters and merges them iteratively. During each merge step, vocabulary size changes and new subword tokens are formed.

Why Subword Tokens Help

Subword tokenization reduces the out-of-vocabulary problem. Even if a word does not appear during training, it can still be represented using smaller known components. This improves model generalization and reduces vocabulary size.

In [2]:
#question 2.2
from collections import Counter

TOY_CORPUS = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new".split()

def word_to_symbols(word: str):
    return list(word) + ["_"]  # end-of-word marker

def get_pair_counts(corpus_syms):
    pairs = Counter()
    for syms in corpus_syms:
        for i in range(len(syms) - 1):
            pairs[(syms[i], syms[i + 1])] += 1
    return pairs

def merge_pair(corpus_syms, pair):
    a, b = pair
    merged = a + b
    new_corpus = []
    for syms in corpus_syms:
        i = 0
        out = []
        while i < len(syms):
            if i < len(syms) - 1 and syms[i] == a and syms[i + 1] == b:
                out.append(merged)
                i += 2
            else:
                out.append(syms[i])
                i += 1
        new_corpus.append(out)
    return new_corpus

def vocab_from_corpus(corpus_syms):
    v = set()
    for syms in corpus_syms:
        v.update(syms)
    return v

def bpe_train(words, merges=10):
    corpus_syms = [word_to_symbols(w) for w in words]
    merges_done = []

    for step in range(1, merges + 1):
        pair_counts = get_pair_counts(corpus_syms)
        if not pair_counts:
            break

        top_pair, top_count = pair_counts.most_common(1)[0]
        corpus_syms = merge_pair(corpus_syms, top_pair)
        merges_done.append(top_pair)

        vocab = vocab_from_corpus(corpus_syms)
        print(f"Step {step:02d}: merge {top_pair} (count={top_count}), vocab_size={len(vocab)}")

    return merges_done

def segment_word(word, merges_done):
    syms = word_to_symbols(word)
    for pair in merges_done:
        syms = merge_pair([syms], pair)[0]
    return syms

merges_done = bpe_train(TOY_CORPUS, merges=20)

print("\nSegmentation examples:")
for w in ["new", "newer", "lowest", "widest", "newestest"]:
    print(f"{w:10s} -> {segment_word(w, merges_done)}")


Step 01: merge ('e', 'r') (count=9), vocab_size=11
Step 02: merge ('er', '_') (count=9), vocab_size=11
Step 03: merge ('n', 'e') (count=8), vocab_size=11
Step 04: merge ('ne', 'w') (count=8), vocab_size=11
Step 05: merge ('l', 'o') (count=7), vocab_size=10
Step 06: merge ('lo', 'w') (count=7), vocab_size=10
Step 07: merge ('new', 'er_') (count=6), vocab_size=11
Step 08: merge ('low', '_') (count=5), vocab_size=12
Step 09: merge ('w', 'i') (count=3), vocab_size=11
Step 10: merge ('wi', 'd') (count=3), vocab_size=10
Step 11: merge ('wid', 'er_') (count=3), vocab_size=9
Step 12: merge ('low', 'e') (count=2), vocab_size=8
Step 13: merge ('lowe', 's') (count=2), vocab_size=7
Step 14: merge ('lowes', 't') (count=2), vocab_size=6
Step 15: merge ('lowest', '_') (count=2), vocab_size=6
Step 16: merge ('new', '_') (count=2), vocab_size=5

Segmentation examples:
new        -> ['new_']
newer      -> ['newer_']
lowest     -> ['lowest_']
widest     -> ['wid', 'e', 's', 't', '_']
newestest  -> ['new'

#Question 2.3

Byte Pair Encoding forms subword tokens by merging frequently occurring character pairs. During the experiment, the algorithm learned meaningful suffixes and repeated word segments. One advantage of BPE is that it helps represent unknown words using smaller subword units. This allows language models to process rare words more effectively. Another advantage is that it reduces vocabulary size while preserving important linguistic patterns. However, BPE sometimes produces subwords that do not follow natural linguistic boundaries. Additionally, subword tokenization can increase sequence length, which may increase computational cost. Overall, BPE provides a good balance between character-level and word-level tokenization.

In [3]:
#question 2.3
from collections import Counter
import re

PARA = (
    "Natural language processing helps computers understand text. "
    "In real data, many words are rare or misspelled. "
    "Subword models make training more robust. "
    "They can represent unseen words using smaller pieces. "
    "This improves generalization across new documents."
)

def tokenize(text):
    # keep words only (lowercase)
    return re.findall(r"[A-Za-z']+", text.lower())

def word_to_symbols(word):
    return list(word) + ["_"]

def get_pair_counts(corpus_syms):
    pairs = Counter()
    for syms in corpus_syms:
        for i in range(len(syms) - 1):
            pairs[(syms[i], syms[i + 1])] += 1
    return pairs

def merge_pair(corpus_syms, pair):
    a, b = pair
    merged = a + b
    new_corpus = []
    for syms in corpus_syms:
        i = 0
        out = []
        while i < len(syms):
            if i < len(syms) - 1 and syms[i] == a and syms[i + 1] == b:
                out.append(merged); i += 2
            else:
                out.append(syms[i]); i += 1
        new_corpus.append(out)
    return new_corpus

def vocab_from_corpus(corpus_syms):
    v = set()
    for syms in corpus_syms:
        v.update(syms)
    return v

def train_bpe_on_paragraph(text, merges=30):
    words = tokenize(text)
    corpus_syms = [word_to_symbols(w) for w in words]
    merges_done = []

    for _ in range(merges):
        pairs = get_pair_counts(corpus_syms)
        if not pairs:
            break
        top_pair, _cnt = pairs.most_common(1)[0]
        merges_done.append(top_pair)
        corpus_syms = merge_pair(corpus_syms, top_pair)

    return merges_done, corpus_syms

def segment_word(word, merges_done):
    syms = word_to_symbols(word.lower())
    for pair in merges_done:
        syms = merge_pair([syms], pair)[0]
    return syms

merges_done, final_corpus = train_bpe_on_paragraph(PARA, merges=30)

print("Top 5 merges learned:", merges_done[:5])

vocab = sorted(vocab_from_corpus(final_corpus), key=len, reverse=True)
print("5 longest tokens:", vocab[:5])

print("\nSegment 5 words:")
for w in ["processing", "misspelled", "robust", "generalization", "unseen"]:
    print(w, "->", segment_word(w, merges_done))


Top 5 merges learned: [('s', '_'), ('r', 'e'), ('i', 'n'), ('r', 'a'), ('a', 'n')]
5 longest tokens: ['words_', 'ing_', 'ral', 'pro', 'wor']

Segment 5 words:
processing -> ['pro', 'ce', 'ss', 'ing_']
misspelled -> ['m', 'i', 'ss', 'p', 'el', 'l', 'e', 'd_']
robust -> ['ro', 'b', 'us', 't_']
generalization -> ['g', 'en', 'e', 'ral', 'i', 'z', 'at', 'i', 'o', 'n', '_']
unseen -> ['un', 's', 'e', 'en', '_']


#Question 3
Bayes Rule Applied to Text

1.P(c)

 This represents the probability of a class occurring before observing any document. It is also called prior probability.

 P(d | c)

 This represents the probability of observing a document given that it belongs to a specific class.

 P(c | d)

 This represents the probability that a document belongs to a class after observing the document.

2.Why P(d) Is Ignored

When comparing class probabilities, P(d) remains constant for all classes. Therefore, classification only depends on maximizing the numerator P(d|c)P(c).

#Question 4

When you build a Naive Bayes text classifier, you need probabilities like:

𝑃
(
word
∣
class
)
P(word∣class)

Example:
𝑃
(
fun
∣
negative
)
P(fun∣negative)

The problem

If a word never appeared in the training data for that class, then:

𝑐
𝑜
𝑢
𝑛
𝑡
(
word
,
class
)
=
0
⇒
𝑃
(
word
∣
class
)
=
0
count(word,class)=0⇒P(word∣class)=0

And when you multiply probabilities for a document, one zero makes the whole class probability zero, even if other words strongly support that class.

So we use Add-1 smoothing to avoid zero probabilities.

Add-1 (Laplace) smoothing formula
𝑃
(
𝑤
∣
𝑐
)
=
𝑐
𝑜
𝑢
𝑛
𝑡
(
𝑤
,
𝑐
)
+
1
𝑁
𝑐
+
∣
𝑉
∣
P(w∣c)=
N
c
	​

+∣V∣
count(w,c)+1
	​


Where:

𝑐
𝑜
𝑢
𝑛
𝑡
(
𝑤
,
𝑐
)
count(w,c) = how many times word
𝑤
w appears in class
𝑐
c

𝑁
𝑐
N
c
	​

 = total number of word tokens in class
𝑐
c

∣
𝑉
∣
∣V∣ = vocabulary size (unique words overall)

Given in your question

Negative class total tokens:

𝑁
𝑛
𝑒
𝑔
=
14
N
neg
	​

=14

Vocabulary size:

∣
𝑉
∣
=
20
∣V∣=20

So the denominator becomes:

𝑁
𝑛
𝑒
𝑔
+
∣
𝑉
∣
=
14
+
20
=
34
N
neg
	​

+∣V∣=14+20=34

This 34 is used for every word in the negative class once smoothing is applied.

1) Compute
𝑃
(
predictable
∣
𝑛
𝑒
𝑔
)
P(predictable∣neg)

Given:

𝑐
𝑜
𝑢
𝑛
𝑡
(
predictable
,
𝑛
𝑒
𝑔
)
=
2
count(predictable,neg)=2

Apply smoothing:

𝑃
(
predictable
∣
𝑛
𝑒
𝑔
)
=
2
+
1
34
=
3
34
P(predictable∣neg)=
34
2+1
	​

=
34
3
	​


Final:

𝑃
(
predictable
∣
𝑛
𝑒
𝑔
)
=
3
34
≈
0.0882
P(predictable∣neg)=
34
3
	​

≈0.0882
2) Compute
𝑃
(
fun
∣
𝑛
𝑒
𝑔
)
P(fun∣neg)

Given:

𝑐
𝑜
𝑢
𝑛
𝑡
(
fun
,
𝑛
𝑒
𝑔
)
=
0
count(fun,neg)=0

Without smoothing it would be:

0
/
14
=
0
0/14=0

But with smoothing:

𝑃
(
fun
∣
𝑛
𝑒
𝑔
)
=
0
+
1
34
=
1
34
P(fun∣neg)=
34
0+1
	​

=
34
1
	​


Final:

𝑃
(
fun
∣
𝑛
𝑒
𝑔
)
=
1
34
≈
0.0294
P(fun∣neg)=
34
1
	​

≈0.0294
Why the denominator adds
∣
𝑉
∣
∣V∣

Because Add-1 smoothing adds 1 extra count to every word in the vocabulary, so you add:

1
×
∣
𝑉
∣
1×∣V∣

to the total count.

So total token mass increases from:

14
→
14
+
20
=
34
14→14+20=34

##Question5

#Tokenization Programming
Paragraph Used

"I can’t believe it’s not butter! In Kansas City, I’m reading state-of-the-art NLP papers. Don’t split ‘New York’ into two tokens. Tokenization is harder than it looks?"

#Naive Tokenization

Naive tokenization splits text based on spaces. This method fails to separate punctuation and contractions properly.

#Manual Tokenization

Manual tokenization separates punctuation marks and contractions using rule-based processing. This produces cleaner tokens and improves text representation.

#Tool-Based Tokenization

spaCy tokenizer automatically processes punctuation and contractions. It provides more standardized token outputs but may sometimes split hyphenated words differently.

#Multiword Expressions

Examples:

 Kansas City

 New York

 State of the art

These expressions must be treated as single tokens because splitting them changes their meaning.

#Reflection

Tokenization is challenging because punctuation, contractions, and multiword expressions create ambiguity in word boundaries. English tokenization requires handling contractions such as "can't" and "I'm". Multiword expressions also require contextual understanding to preserve meaning. Compared to English, languages with complex word formation rules may require more advanced tokenization methods. Tool-based tokenizers improve accuracy but still depend on predefined linguistic rules. Overall, tokenization plays a critical role in NLP pipeline performance.

In [4]:
#question 5
import re

TEXT = (
    "I can’t believe it’s not butter! In Kansas City, I’m reading state-of-the-art NLP papers. "
    "Don’t split ‘New York’ into two tokens. Tokenization is harder than it looks?"
)

def normalize_quotes(text):
    # convert curly quotes to straight quotes so rules behave consistently
    return (text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"'))

def naive_tokenize(text):
    return text.split()

def manual_tokenize(text):
    text = normalize_quotes(text)

    # separate punctuation into its own token
    text = re.sub(r"([.,!?;:()])", r" \1 ", text)

    # split common contractions (simple rules)
    text = re.sub(r"\b(can)('t)\b", r"\1 \2", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(do)('n't)\b", r"\1 \2", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(I)('m)\b", r"\1 \2", text)
    text = re.sub(r"\b(it)('s)\b", r"\1 \2", text, flags=re.IGNORECASE)

    # collapse extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text.split(" ")

def spacy_tokenize(text):
    try:
        import spacy
        nlp = spacy.blank("en")
        return [t.text for t in nlp(text)]
    except Exception as e:
        return ["spaCy not available in this runtime:", str(e)]

print("TEXT:\n", TEXT)

print("\nNaive tokens:")
print(naive_tokenize(TEXT))

print("\nManual tokens:")
print(manual_tokenize(TEXT))

print("\nspaCy tokens:")
print(spacy_tokenize(TEXT))

print("\nMWEs examples:")
print(["Kansas City", "New York", "state-of-the-art"])


TEXT:
 I can’t believe it’s not butter! In Kansas City, I’m reading state-of-the-art NLP papers. Don’t split ‘New York’ into two tokens. Tokenization is harder than it looks?

Naive tokens:
['I', 'can’t', 'believe', 'it’s', 'not', 'butter!', 'In', 'Kansas', 'City,', 'I’m', 'reading', 'state-of-the-art', 'NLP', 'papers.', 'Don’t', 'split', '‘New', 'York’', 'into', 'two', 'tokens.', 'Tokenization', 'is', 'harder', 'than', 'it', 'looks?']

Manual tokens:
['I', 'can', "'t", 'believe', 'it', "'s", 'not', 'butter', '!', 'In', 'Kansas', 'City', ',', 'I', "'m", 'reading', 'state-of-the-art', 'NLP', 'papers', '.', "Don't", 'split', "'New", "York'", 'into', 'two', 'tokens', '.', 'Tokenization', 'is', 'harder', 'than', 'it', 'looks', '?']

spaCy tokens:
['I', 'ca', 'n’t', 'believe', 'it', '’s', 'not', 'butter', '!', 'In', 'Kansas', 'City', ',', 'I', '’m', 'reading', 'state', '-', 'of', '-', 'the', '-', 'art', 'NLP', 'papers', '.', 'Do', 'n’t', 'split', '‘', 'New', 'York', '’', 'into', 'two', 'tok